# 01 · Acquire

This notebook downloads every source and saves it in `data/raw/`. Show your download log at the end.

In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

In [2]:
retry = Retry(total=5, backoff_factor=0.5, status_forcelist=[429, 500, 502, 503, 504])
session = requests.Session()
session.mount("http://", HTTPAdapter(max_retries=retry))
session.mount("https://", HTTPAdapter(max_retries=retry))
session.headers["User-Agent"] = "ESMT MAAI data wrangling project, group N (YOUR-EMAIL-HERE)"

log = []

## Source 1

In [3]:
OECD_URL = "https://sdmx.oecd.org/public/rest/data/OECD.SDD.STES,DSD_STES@DF_FINMARK/.A.SHARE......"
oecd_file = RAW / "oecd_share_prices.csv"

if not oecd_file.exists():
    try:
        r = session.get(OECD_URL, params={"startPeriod": 2014, "endPeriod": 2024, "format": "csvfile"}, timeout=30)
        r.raise_for_status()
        oecd_file.write_bytes(r.content)
        outcome = "downloaded"
    except requests.HTTPError as e:
        outcome = f"failed with {e.response.status_code}"
else:
    outcome = "already in data/raw"

log.append({"source": "OECD", "file": oecd_file.name, "url": OECD_URL, "outcome": outcome})
outcome

'already in data/raw'

## Source 2

In [4]:
WB_URL = "https://api.worldbank.org/v2/country/all/indicator/FP.CPI.TOTL.ZG"

page = 1
pages = 1

while page <= pages:
    path = RAW / f"worldbank_inflation_page_{page:02d}.json"
    if not path.exists():
        try:
            r = session.get(WB_URL, params={"format": "json", "date": "2014:2024", "per_page": 1000, "page": page}, timeout=30)
            r.raise_for_status()
            path.write_bytes(r.content)
            outcome = "downloaded"
            time.sleep(0.5)
        except requests.HTTPError as e:
            outcome = f"failed with {e.response.status_code}"
    else:
        outcome = "already in data/raw"

    log.append({"source": "World Bank", "file": path.name, "url": f"{WB_URL}?page={page}", "outcome": outcome})
    if not path.exists():
        break

    with open(path) as f:
        pages = json.load(f)[0]["pages"]
    page += 1

print(pages, "pages")

3 pages


## Source 3

In [5]:
WIKI_URL = "https://en.wikipedia.org/wiki/ISO_3166-1"
wiki_file = RAW / "wikipedia_iso_3166_1.html"

if not wiki_file.exists():
    try:
        r = session.get(WIKI_URL, timeout=30)
        r.raise_for_status()
        wiki_file.write_bytes(r.content)
        outcome = "downloaded"
    except requests.HTTPError as e:
        outcome = f"failed with {e.response.status_code}"
else:
    outcome = "already in data/raw"

log.append({"source": "Wikipedia", "file": wiki_file.name, "url": WIKI_URL, "outcome": outcome})
outcome

'already in data/raw'

## Download log

In [6]:
download_log = pd.DataFrame(log)
download_log["checked_on"] = pd.Timestamp.today().strftime("%Y-%m-%d")
download_log.to_csv(RAW / "download_log.csv", index=False)
download_log

,source,file,url,outcome,checked_on
0,OECD,oecd_share_prices.csv,https://sdmx.oecd.org/public/rest/data/OECD.SD...,already in data/raw,2026-11-17
1,World Bank,worldbank_inflation_page_01.json,https://api.worldbank.org/v2/country/all/indic...,already in data/raw,2026-11-17
2,World Bank,worldbank_inflation_page_02.json,https://api.worldbank.org/v2/country/all/indic...,already in data/raw,2026-11-17
3,World Bank,worldbank_inflation_page_03.json,https://api.worldbank.org/v2/country/all/indic...,already in data/raw,2026-11-17
4,Wikipedia,wikipedia_iso_3166_1.html,https://en.wikipedia.org/wiki/ISO_3166-1,already in data/raw,2026-11-17


In [7]:
sorted(p.name for p in RAW.iterdir())

['.gitkeep',
 'download_log.csv',
 'oecd_share_prices.csv',
 'wikipedia_iso_3166_1.html',
 'worldbank_inflation_page_01.json',
 'worldbank_inflation_page_02.json',
 'worldbank_inflation_page_03.json']